# LeetCode 1538 — Guess the Majority in a Hidden Array bằng Python

Link đề: https://leetcode.com/problems/guess-the-majority-in-a-hidden-array/ · Độ khó: **Medium** · Chủ đề: Array, Math, Interactive, Boyer–Moore Majority Vote Algorithm

> **Ghi chú Premium:** đây là bài Premium (🔒). Đề bài, chữ ký và giới hạn số lần gọi trong notebook được viết theo hiểu biết về bài, đối chiếu với trang doocs (https://leetcode.doocs.org/en/lc/1538/). Nếu có tài khoản Premium, nên đối chiếu lại với đề gốc. Lớp `ArrayReader` trong notebook là **bản giả lập** tự viết để chạy thử.

Skeleton LeetCode (theo doocs):

```python
# """
# This is the ArrayReader's API interface.
# You should not implement it, or speculate about its implementation
# """
# class ArrayReader(object):
#    # Compares 4 different elements in the array
#    # return 4 if the values of the 4 elements are the same (0 or 1).
#    # return 2 if three elements have a value equal to 0 and one element has value equal to 1 or vice versa.
#    # return 0 : if two element have a value equal to 0 and two elements have a value equal to 1.
#    def query(self, a: int, b: int, c: int, d: int) -> int:
#
#    # Returns the length of the array
#    def length(self) -> int:

class Solution:
    def guessMajority(self, reader: "ArrayReader") -> int:
        ...
```

## 1. Mục tiêu học

1. Làm quen bài **tương tác** có ngân sách truy vấn (`query` tối đa `2n` lần) và tự dựng **bộ giả lập** `ArrayReader` có đếm lần gọi, kiểm tra tham số.
2. Hiểu insight "**giữ cố định 3 phần tử, thay phần tử thứ 4**": kết quả `query` giống nhau ⇔ phần tử thay vào **cùng giá trị** với phần tử cũ.
3. Chia các chỉ số thành **hai nhóm** (cùng / khác giá trị với `nums[3]`) và trả về đại diện nhóm lớn hơn, `-1` nếu hoà — không cần biết nhóm nào là 0 hay 1.
4. Viết lời giải doocs (`n + 1` truy vấn), cải tiến còn `n` truy vấn, và (bài tập) còn `n − 1` truy vấn.
5. Test đúng kiểu bài **nhiều đáp án**: dùng **hàm kiểm tra hợp lệ** thay vì so với một đáp án cố định.

## 2. Đề bài & ví dụ

Mảng ẩn `nums` chỉ gồm 0 và 1. Ta chỉ truy cập qua `reader` với hai hàm:

- `query(a, b, c, d)` với `0 ≤ a < b < c < d < length()`: xét **phân bố** giá trị của 4 phần tử, trả về
  - `4` nếu cả 4 phần tử bằng nhau (đều 0 hoặc đều 1);
  - `2` nếu ba phần tử bằng 0 và một bằng 1, **hoặc ngược lại**;
  - `0` nếu hai phần tử bằng 0 và hai phần tử bằng 1.
- `length()`: độ dài mảng.

Được gọi `query` **tối đa `2n`** lần (`n = length()`). Trả về **một chỉ số bất kỳ** của giá trị xuất hiện nhiều nhất; nếu số 0 và số 1 **bằng nhau** thì trả về `-1`.

**Ví dụ 1:** `nums = [0,0,1,0,1,1,1,1]` → `5` (các chỉ số `2, 4, 6, 7` cũng đúng). Chuỗi gọi mẫu: `query(0,1,2,3)` → `2`, `query(4,5,6,7)` → `4`.

**Ví dụ 2:** `nums = [0,0,1,1,0]` → `0` (bất kỳ chỉ số nào của số 0: `0, 1, 4`).

**Ví dụ 3:** `nums = [1,0,1,0,1,0,1,0]` → `-1`.

**Ràng buộc:** `5 ≤ nums.length ≤ 10⁵`, `0 ≤ nums[i] ≤ 1`.

**Follow-up:** số lần gọi tối thiểu để tìm phần tử đa số là bao nhiêu? (bài tập 5)

### Bộ giả lập `ArrayReader` (bản giả lập tự viết — không phải code của LeetCode)

- Giữ mảng ẩn, đếm `calls`, giới hạn mặc định `2n`; vượt giới hạn ⇒ `exceeded = True` và ném `CallLimitExceeded`.
- Kiểm tra `0 ≤ a < b < c < d < n`, sai ⇒ `ValueError`.
- `valid_answer(nums, ans)`: kiểm tra đáp án hợp lệ (nhiều đáp án đúng!).
- `run(cls, nums)`: chạy lời giải, bắt mọi lỗi giả lập và trả `(đáp án, số lần gọi, trạng thái)`.

In [1]:
import itertools
import random
import time
from typing import List


class CallLimitExceeded(Exception):
    pass


class ArrayReader:
    """BẢN GIẢ LẬP ArrayReader của LeetCode 1538 (tự viết để test)."""

    def __init__(self, nums: List[int], limit=None, log=False):
        self._nums = list(nums)
        self.limit = 2 * len(nums) if limit is None else limit
        self.calls = 0
        self.exceeded = False
        self.log = log

    def query(self, a: int, b: int, c: int, d: int) -> int:
        n = len(self._nums)
        if not (0 <= a < b < c < d < n):
            raise ValueError(f"tham số sai: query({a}, {b}, {c}, {d}) với n={n}")
        self.calls += 1
        if self.calls > self.limit:
            self.exceeded = True
            raise CallLimitExceeded(f"vượt quá {self.limit} lần gọi query")
        ones = self._nums[a] + self._nums[b] + self._nums[c] + self._nums[d]
        res = 4 if ones in (0, 4) else (2 if ones in (1, 3) else 0)
        if self.log:
            print(f"  query({a}, {b}, {c}, {d}) -> {res}")
        return res

    def length(self) -> int:
        return len(self._nums)


def valid_answer(nums, ans):
    """Đáp án hợp lệ? Hoà -> phải là -1; ngược lại là chỉ số bất kỳ của giá trị đa số."""
    ones = sum(nums)
    zeros = len(nums) - ones
    if ones == zeros:
        return ans == -1
    major = 1 if ones > zeros else 0
    return isinstance(ans, int) and 0 <= ans < len(nums) and nums[ans] == major


def run(solution_cls, nums, limit=None, log=False):
    reader = ArrayReader(nums, limit, log)
    try:
        ans = solution_cls().guessMajority(reader)
    except CallLimitExceeded as e:
        return None, reader.calls, "VƯỢT GIỚI HẠN: " + str(e)
    except ValueError as e:
        return None, reader.calls, "GỌI SAI: " + str(e)
    return ans, reader.calls, ("HỢP LỆ" if valid_answer(nums, ans) else "SAI")


examples = [[0, 0, 1, 0, 1, 1, 1, 1], [0, 0, 1, 1, 0], [1, 0, 1, 0, 1, 0, 1, 0]]
rd = ArrayReader(examples[0], log=True)
rd.query(0, 1, 2, 3)
rd.query(4, 5, 6, 7)
for nums in examples:
    print(nums, "| đáp án hợp lệ của đề:", [i for i in range(len(nums)) if valid_answer(nums, i)] or [-1])

  query(0, 1, 2, 3) -> 2
  query(4, 5, 6, 7) -> 4
[0, 0, 1, 0, 1, 1, 1, 1] | đáp án hợp lệ của đề: [2, 4, 5, 6, 7]
[0, 0, 1, 1, 0] | đáp án hợp lệ của đề: [0, 1, 4]
[1, 0, 1, 0, 1, 0, 1, 0] | đáp án hợp lệ của đề: [-1]


## 3. Kiến thức nền

### 3.1 Bảng kết quả `query` theo số bit 1

Gọi `c` = số phần tử bằng 1 trong 4 phần tử. Kết quả chỉ phụ thuộc `c`:

| c (số bit 1) | 0 | 1 | 2 | 3 | 4 |
|--------------|---|---|---|---|---|
| query | 4 | 2 | 0 | 2 | 4 |

Quan trọng: **hai giá trị `c` liền nhau luôn cho kết quả khác nhau** (4≠2, 2≠0, 0≠2, 2≠4). Đây là chìa khoá của bài.

In [2]:
def f(c):
    return 4 if c in (0, 4) else (2 if c in (1, 3) else 0)

print({c: f(c) for c in range(5)})
print("c và c+1 luôn khác kết quả:", all(f(c) != f(c + 1) for c in range(4)))

{0: 4, 1: 2, 2: 0, 3: 2, 4: 4}
c và c+1 luôn khác kết quả: True


### 3.2 Đối xứng đảo bit

Đảo mọi bit (0 ↔ 1) biến `c` thành `4 − c`, mà `f(c) = f(4 − c)` ⇒ **mọi** `query` cho cùng kết quả trên `nums` và trên mảng đảo bit. Hệ quả: ta **không bao giờ** biết được phần tử nào là 0, phần tử nào là 1 — chỉ biết **cái nào bằng cái nào**. May mắn là đề chỉ hỏi chỉ số của giá trị đa số.

In [3]:
nums = [0, 0, 1, 0, 1, 1, 1, 1]
flipped = [1 - v for v in nums]
same = all(ArrayReader(nums, 10**9).query(*q) == ArrayReader(flipped, 10**9).query(*q)
           for q in itertools.combinations(range(8), 4))
print("Mọi bộ 4 cho cùng kết quả trên nums và mảng đảo bit:", same)

Mọi bộ 4 cho cùng kết quả trên nums và mảng đảo bit: True


### 3.3 `itertools.combinations` — sinh mọi bộ chỉ số tăng dần

`combinations(range(n), 4)` sinh mọi bộ `(a, b, c, d)` với `a < b < c < d` — đúng điều kiện của `query`. Có `C(n, 4)` bộ.

In [4]:
print(list(itertools.combinations(range(5), 4)))
from math import comb
print("C(8,4) =", comb(8, 4), "| C(100,4) =", comb(100, 4))

[(0, 1, 2, 3), (0, 1, 2, 4), (0, 1, 3, 4), (0, 2, 3, 4), (1, 2, 3, 4)]
C(8,4) = 70 | C(100,4) = 3921225


### 3.4 Đếm hai nhóm và chọn đại diện

Khi đã có nhãn `same[i]` (cùng giá trị với phần tử tham chiếu hay không), số phần tử mỗi nhóm là `a = Σ same`, `b = n − a`. Nhóm lớn hơn là đa số; cần nhớ **một chỉ số** thuộc mỗi nhóm để trả về.

In [5]:
same = [True, True, False, True, False, False, False, False]   # so với nums[3] = 0
a = sum(same); b = len(same) - a
rep_same = same.index(True); rep_diff = same.index(False)
print(f"a={a}, b={b} -> trả về", -1 if a == b else (rep_same if a > b else rep_diff))

a=3, b=5 -> trả về 2


### 3.5 Exception tự định nghĩa để báo vượt ngân sách

Giống bài 1533: `CallLimitExceeded` giúp `run` phân biệt "vượt giới hạn" với "trả lời sai".

In [6]:
try:
    r = ArrayReader([0, 1, 0, 1, 0], limit=1)
    r.query(0, 1, 2, 3)
    r.query(0, 1, 2, 4)
except CallLimitExceeded as e:
    print("Bắt được:", e)
try:
    ArrayReader([0, 1, 0, 1, 0]).query(0, 0, 1, 2)
except ValueError as e:
    print("Bắt được:", e)

Bắt được: vượt quá 1 lần gọi query
Bắt được: tham số sai: query(0, 0, 1, 2) với n=5


## 4. Ý tưởng nền tảng / trực giác

Giữ cố định ba phần tử `0, 1, 2` (tổng bit `s`). So sánh:

- `x = query(0, 1, 2, 3) = f(s + nums[3])`
- `query(0, 1, 2, i) = f(s + nums[i])`

Nếu `nums[i] = nums[3]` thì hai kết quả **bằng nhau**. Nếu khác thì `s + nums[i]` và `s + nums[3]` hơn kém nhau 1 ⇒ kết quả **khác nhau** (mục 3.1). Vậy một truy vấn cho biết `nums[i]` có bằng `nums[3]` hay không!

Tính tay ví dụ 1 `nums = [0,0,1,0,1,1,1,1]`: `s = 1`, `x = f(1 + 0) = 2`.

| i | nums[i] | query(0,1,2,i) | so với x = 2 | nhóm |
|---|---------|----------------|--------------|------|
| 4 | 1 | f(2) = 0 | khác | B |
| 5 | 1 | 0 | khác | B |
| 6 | 1 | 0 | khác | B |
| 7 | 1 | 0 | khác | B |

Còn chỉ số `0, 1, 2` (nằm trong bộ cố định) phải so cách khác — dùng bộ cố định mới `{?, ?, ?, 4}` (mục 6). Kết quả: nhóm A (cùng `nums[3]`) = `{3, 0, 1}`, nhóm B = `{2, 4, 5, 6, 7}` ⇒ đa số là B.

In [7]:
nums = examples[0]
rd = ArrayReader(nums, log=True)
x = rd.query(0, 1, 2, 3)
for i in range(4, len(nums)):
    r = rd.query(0, 1, 2, i)
    print(f"    i={i}: {'cùng' if r == x else 'khác'} giá trị với nums[3]  (thực tế nums[3]={nums[3]}, nums[{i}]={nums[i]})")

  query(0, 1, 2, 3) -> 2
  query(0, 1, 2, 4) -> 0
    i=4: khác giá trị với nums[3]  (thực tế nums[3]=0, nums[4]=1)
  query(0, 1, 2, 5) -> 0
    i=5: khác giá trị với nums[3]  (thực tế nums[3]=0, nums[5]=1)
  query(0, 1, 2, 6) -> 0
    i=6: khác giá trị với nums[3]  (thực tế nums[3]=0, nums[6]=1)
  query(0, 1, 2, 7) -> 0
    i=7: khác giá trị với nums[3]  (thực tế nums[3]=0, nums[7]=1)


## 5. Cách 1 — Brute Force / baseline: hỏi mọi bộ 4 rồi vét cạn

Ý tưởng "ngây thơ" nhưng chắc chắn đúng: hỏi **tất cả** `C(n, 4)` bộ, rồi thử mọi mảng `2ⁿ` khả dĩ, giữ những mảng khớp mọi kết quả. Mọi mảng khớp đều cho cùng cách chia hai nhóm (chỉ khác nhau phép đảo bit), nên lấy một mảng khớp bất kỳ để chọn đáp án.

In [8]:
class SolutionBrute:
    def guessMajority(self, reader: "ArrayReader") -> int:
        n = reader.length()
        answers = {q: reader.query(*q) for q in itertools.combinations(range(n), 4)}
        for bits in itertools.product([0, 1], repeat=n):
            if all(f(bits[a] + bits[b] + bits[c] + bits[d]) == r for (a, b, c, d), r in answers.items()):
                ones = sum(bits)
                if 2 * ones == n:
                    return -1
                major = 1 if 2 * ones > n else 0
                return bits.index(major)
        return -1

for nums in examples[1:]:
    print(nums, "->", run(SolutionBrute, nums, limit=10**9))

[0, 0, 1, 1, 0] -> (0, 5, 'HỢP LỆ')
[1, 0, 1, 0, 1, 0, 1, 0] -> (-1, 70, 'HỢP LỆ')


Đúng nhưng số truy vấn `C(n, 4)` bùng nổ: `n = 8` đã cần 70 > `2n = 16`. Với giới hạn thật, bộ giả lập báo vượt ngay:

In [9]:
for nums in examples:
    print(len(nums), "phần tử:", run(SolutionBrute, nums))

8 phần tử: (None, 17, 'VƯỢT GIỚI HẠN: vượt quá 16 lần gọi query')
5 phần tử: (0, 5, 'HỢP LỆ')
8 phần tử: (None, 17, 'VƯỢT GIỚI HẠN: vượt quá 16 lần gọi query')


**Độ phức tạp:** `C(n,4) = O(n⁴)` truy vấn, `O(2ⁿ · n⁴)` thời gian. **Nút thắt:** hầu hết các truy vấn là thừa — ta chỉ cần biết mỗi phần tử **cùng hay khác** một phần tử tham chiếu, tức `n − 1` mẩu thông tin nhị phân.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight (thay một phần tử):** hai truy vấn chỉ khác nhau **một** chỉ số (`p` thay bằng `q`, ba chỉ số kia giữ nguyên) cho kết quả bằng nhau ⇔ `nums[p] = nums[q]`.

**Áp dụng:**
1. Tham chiếu: phần tử 3. `x = query(0,1,2,3)`.
2. Với `i = 4..n−1`: `query(0,1,2,i) == x` ⇔ `nums[i] == nums[3]`.
3. Với `0, 1, 2`: chọn bộ cố định khác chứa 4: `y = query(0,1,2,4)`; thay `0` bằng `3` → `query(1,2,3,4)`: bằng `y` ⇔ `nums[0] == nums[3]`. Tương tự `query(0,2,3,4)` cho chỉ số 1, `query(0,1,3,4)` cho chỉ số 2.
4. Đếm `a` = số phần tử cùng nhóm với 3 (gồm chính nó), `b` = số còn lại, nhớ `k` = một chỉ số thuộc nhóm B.
5. **Nếu** `a == b` **thì** `-1`; **nếu** `a > b` **thì** trả `3`; **ngược lại** trả `k`.

**Invariant:** sau khi xử lý chỉ số `i`, `a + b` = số chỉ số đã phân loại, `k` (nếu `b > 0`) là một chỉ số đã biết khác giá trị `nums[3]`. Số truy vấn: `1 + (n − 4) + 4 = n + 1 ≤ 2n`.

## 7. Các cách tiếp theo

### Cách 2 — Lời giải doocs (`n + 1` truy vấn)

Từng đoạn:
- `a, b = 1, 0`: chỉ số 3 thuộc nhóm A.
- Vòng `i = 4..n−1`: phân loại bằng bộ cố định `{0,1,2}`.
- Ba truy vấn cuối phân loại `0, 1, 2` bằng bộ cố định `{?,?,3,4}` với `y = query(0,1,2,4)`.

In [10]:
class SolutionDoocs:
    def guessMajority(self, reader: "ArrayReader") -> int:
        n = reader.length()
        x = reader.query(0, 1, 2, 3)
        a, b = 1, 0
        k = 0
        for i in range(4, n):
            if reader.query(0, 1, 2, i) == x:
                a += 1
            else:
                b += 1
                k = i

        y = reader.query(0, 1, 2, 4)
        if reader.query(1, 2, 3, 4) == y:
            a += 1
        else:
            b += 1
            k = 0
        if reader.query(0, 2, 3, 4) == y:
            a += 1
        else:
            b += 1
            k = 1
        if reader.query(0, 1, 3, 4) == y:
            a += 1
        else:
            b += 1
            k = 2

        if a == b:
            return -1
        return 3 if a > b else k

for nums in examples:
    print(nums, "->", run(SolutionDoocs, nums))

[0, 0, 1, 0, 1, 1, 1, 1] -> (2, 9, 'HỢP LỆ')
[0, 0, 1, 1, 0] -> (1, 6, 'HỢP LỆ')
[1, 0, 1, 0, 1, 0, 1, 0] -> (-1, 9, 'HỢP LỆ')


**Độ phức tạp:** `n + 1` truy vấn, Time `O(n)`, Space `O(1)`.

Chú ý giá trị khởi tạo `k = 0`: nếu kết thúc với `b > a` thì chắc chắn `b ≥ 1` nên `k` đã được gán bởi một phần tử nhóm B — khởi tạo chỉ để có giá trị.

### Cách 3 — Dùng lại kết quả đã có (`n` truy vấn) và lưu nhãn

Để ý `y = query(0,1,2,4)` **chính là** truy vấn đã hỏi ở vòng lặp khi `i = 4`. Lưu lại thay vì hỏi lần nữa ⇒ còn `n` truy vấn. Đồng thời lưu mảng nhãn `same[i]` để code rõ ràng hơn (đổi lại `O(n)` bộ nhớ).

In [11]:
class SolutionCached:
    def guessMajority(self, reader: "ArrayReader") -> int:
        n = reader.length()
        same = [False] * n
        same[3] = True
        x = reader.query(0, 1, 2, 3)
        q4 = None
        for i in range(4, n):
            r = reader.query(0, 1, 2, i)
            if i == 4:
                q4 = r                        # = query(0, 1, 2, 4), dùng lại bên dưới
            same[i] = (r == x)
        # thay lần lượt 0, 1, 2 bằng 3 trong bộ (0, 1, 2, 4)
        same[0] = reader.query(1, 2, 3, 4) == q4
        same[1] = reader.query(0, 2, 3, 4) == q4
        same[2] = reader.query(0, 1, 3, 4) == q4
        a = sum(same)
        b = n - a
        if a == b:
            return -1
        return 3 if a > b else same.index(False)

for nums in examples:
    print(nums, "->", run(SolutionCached, nums))

[0, 0, 1, 0, 1, 1, 1, 1] -> (2, 8, 'HỢP LỆ')
[0, 0, 1, 1, 0] -> (0, 5, 'HỢP LỆ')
[1, 0, 1, 0, 1, 0, 1, 0] -> (-1, 8, 'HỢP LỆ')


**Độ phức tạp:** đúng `n` truy vấn, Time `O(n)`, Space `O(n)` (có thể bỏ mảng `same`, chỉ giữ `a, b, k` như cách 2 để về `O(1)`).

### Cách 4 — Kiểm tra "tham chiếu" bằng mảng thật (chỉ để hiểu): khôi phục `nums` tới mức đảo bit

Từ nhãn `same[i]`, đặt `rec[i] = 0` nếu cùng nhóm với 3, `1` nếu khác. Khi đó `rec` bằng `nums` **hoặc** bằng mảng đảo bit của `nums` (mục 3.2). Đây là cách nhìn "giải mã" giúp viết test: mọi truy vấn trên `rec` phải cho cùng kết quả như trên `nums`.

In [12]:
def recover(reader):
    n = reader.length()
    x = reader.query(0, 1, 2, 3)
    rec = [0] * n
    q4 = None
    for i in range(4, n):
        r = reader.query(0, 1, 2, i)
        q4 = r if i == 4 else q4
        rec[i] = 0 if r == x else 1
    rec[0] = 0 if reader.query(1, 2, 3, 4) == q4 else 1
    rec[1] = 0 if reader.query(0, 2, 3, 4) == q4 else 1
    rec[2] = 0 if reader.query(0, 1, 3, 4) == q4 else 1
    return rec

for nums in examples:
    rec = recover(ArrayReader(nums))
    flip = [1 - v for v in nums]
    print(nums, "-> khôi phục", rec, "| bằng nums" if rec == nums else ("| bằng đảo bit" if rec == flip else "| SAI"))

[0, 0, 1, 0, 1, 1, 1, 1] -> khôi phục [0, 0, 1, 0, 1, 1, 1, 1] | bằng nums
[0, 0, 1, 1, 0] -> khôi phục [1, 1, 0, 0, 1] | bằng đảo bit
[1, 0, 1, 0, 1, 0, 1, 0] -> khôi phục [1, 0, 1, 0, 1, 0, 1, 0] | bằng nums


## 8. TRACE từng bước

Trace cách 2 trên ví dụ 2 `nums = [0,0,1,1,0]` (n = 5, giới hạn 10 truy vấn):

| Bước | Truy vấn | Giá trị 4 phần tử | Kết quả | So với | Kết luận | a | b | k |
|------|----------|-------------------|---------|--------|----------|---|---|---|
| 0 | (0,1,2,3) | 0,0,1,1 | 0 | – | x = 0 | 1 | 0 | 0 |
| 1 | (0,1,2,4) | 0,0,1,0 | 2 | x = 0 | 4 khác 3 | 1 | 1 | 4 |
| 2 | (0,1,2,4) | 0,0,1,0 | 2 | – | y = 2 | 1 | 1 | 4 |
| 3 | (1,2,3,4) | 0,1,1,0 | 0 | y = 2 | 0 khác 3 | 1 | 2 | 0 |
| 4 | (0,2,3,4) | 0,1,1,0 | 0 | y = 2 | 1 khác 3 | 1 | 3 | 1 |
| 5 | (0,1,3,4) | 0,0,1,0 | 2 | y = 2 | 2 cùng 3 | 2 | 3 | 1 |

`a = 2 < b = 3` ⇒ trả `k = 1`; `nums[1] = 0` là giá trị đa số ✔ (đề cho `0`, cũng hợp lệ).

In [13]:
def trace_doocs(nums):
    reader = ArrayReader(nums)
    n = reader.length()
    x = reader.query(0, 1, 2, 3)
    a, b, k = 1, 0, 0
    print(f"query(0,1,2,3) = {x} -> x | a={a} b={b} k={k}")
    for i in range(4, n):
        r = reader.query(0, 1, 2, i)
        if r == x:
            a += 1
        else:
            b += 1; k = i
        print(f"query(0,1,2,{i}) = {r} {'==' if r == x else '!='} x -> chỉ số {i} {'cùng' if r == x else 'khác'} nhóm 3 | a={a} b={b} k={k}")
    y = reader.query(0, 1, 2, 4)
    print(f"query(0,1,2,4) = {y} -> y")
    for idx, q in [(0, (1, 2, 3, 4)), (1, (0, 2, 3, 4)), (2, (0, 1, 3, 4))]:
        r = reader.query(*q)
        if r == y:
            a += 1
        else:
            b += 1; k = idx
        print(f"query{q} = {r} {'==' if r == y else '!='} y -> chỉ số {idx} {'cùng' if r == y else 'khác'} nhóm 3 | a={a} b={b} k={k}")
    ans = -1 if a == b else (3 if a > b else k)
    print(f"Kết quả: {ans} | số truy vấn: {reader.calls} / {reader.limit} | hợp lệ: {valid_answer(nums, ans)}")
    return ans

trace_doocs([0, 0, 1, 1, 0])

query(0,1,2,3) = 0 -> x | a=1 b=0 k=0
query(0,1,2,4) = 2 != x -> chỉ số 4 khác nhóm 3 | a=1 b=1 k=4
query(0,1,2,4) = 2 -> y
query(1, 2, 3, 4) = 0 != y -> chỉ số 0 khác nhóm 3 | a=1 b=2 k=0
query(0, 2, 3, 4) = 0 != y -> chỉ số 1 khác nhóm 3 | a=1 b=3 k=1
query(0, 1, 3, 4) = 2 == y -> chỉ số 2 cùng nhóm 3 | a=2 b=3 k=1
Kết quả: 1 | số truy vấn: 6 / 10 | hợp lệ: True


1

In [14]:
print("Ví dụ 3 (hoà):")
trace_doocs(examples[2])

Ví dụ 3 (hoà):
query(0,1,2,3) = 0 -> x | a=1 b=0 k=0
query(0,1,2,4) = 2 != x -> chỉ số 4 khác nhóm 3 | a=1 b=1 k=4
query(0,1,2,5) = 0 == x -> chỉ số 5 cùng nhóm 3 | a=2 b=1 k=4
query(0,1,2,6) = 2 != x -> chỉ số 6 khác nhóm 3 | a=2 b=2 k=6
query(0,1,2,7) = 0 == x -> chỉ số 7 cùng nhóm 3 | a=3 b=2 k=6
query(0,1,2,4) = 2 -> y
query(1, 2, 3, 4) = 0 != y -> chỉ số 0 khác nhóm 3 | a=3 b=3 k=0
query(0, 2, 3, 4) = 2 == y -> chỉ số 1 cùng nhóm 3 | a=4 b=3 k=0
query(0, 1, 3, 4) = 0 != y -> chỉ số 2 khác nhóm 3 | a=4 b=4 k=2
Kết quả: -1 | số truy vấn: 9 / 16 | hợp lệ: True


-1

## 9. Code nộp LeetCode

Bản nộp chính — đúng chữ ký doocs `guessMajority(self, reader: "ArrayReader") -> int`, dùng lại truy vấn `(0,1,2,4)` để chỉ tốn `n` truy vấn và `O(1)` bộ nhớ. Không tự định nghĩa `ArrayReader` khi nộp.

In [15]:
class Solution:
    def guessMajority(self, reader: "ArrayReader") -> int:
        n = reader.length()
        x = reader.query(0, 1, 2, 3)          # tham chiếu: phần tử 3
        a, b, k = 1, 0, -1                    # a: cùng giá trị nums[3]; b: khác; k: 1 chỉ số nhóm b
        y = None
        for i in range(4, n):
            r = reader.query(0, 1, 2, i)
            if i == 4:
                y = r                         # = query(0, 1, 2, 4)
            if r == x:
                a += 1
            else:
                b += 1
                k = i
        # phân loại 0, 1, 2: thay từng chỉ số bằng 3 trong bộ (0, 1, 2, 4)
        for idx, q in ((0, (1, 2, 3, 4)), (1, (0, 2, 3, 4)), (2, (0, 1, 3, 4))):
            if reader.query(*q) == y:
                a += 1
            else:
                b += 1
                k = idx
        if a == b:
            return -1
        return 3 if a > b else k

for nums in examples:
    print(nums, "->", run(Solution, nums))

[0, 0, 1, 0, 1, 1, 1, 1] -> (2, 8, 'HỢP LỆ')
[0, 0, 1, 1, 0] -> (1, 5, 'HỢP LỆ')
[1, 0, 1, 0, 1, 0, 1, 0] -> (-1, 8, 'HỢP LỆ')


## 10. Vì sao thuật toán đúng

**Bổ đề thay thế.** Cho bộ ba cố định `T` (tổng bit `s`) và hai chỉ số `p, q ∉ T`. Khi đó `query(T ∪ {p}) = f(s + nums[p])` và `query(T ∪ {q}) = f(s + nums[q])`. Nếu `nums[p] = nums[q]` thì hai kết quả bằng nhau. Nếu khác, hai đối số hơn kém nhau đúng 1, và `f` nhận giá trị khác nhau trên hai số nguyên liên tiếp (bảng 3.1) ⇒ kết quả khác nhau. Vậy "bằng nhau" ⇔ "cùng giá trị".

**Phân loại đầy đủ.** Chỉ số `i ≥ 4`: `T = {0,1,2}`, `p = 3`, `q = i`. Chỉ số `0`: `T = {1,2,4}`, `p = 0`, `q = 3` (so `query(0,1,2,4)` với `query(1,2,3,4)`); tương tự cho `1` (`T = {0,2,4}`) và `2` (`T = {0,1,4}`). Mọi chỉ số đều được so với phần tử 3 ⇒ `a` = số phần tử bằng `nums[3]`, `b` = số phần tử khác.

**Đáp án.** Hai nhóm đúng là "các số 0" và "các số 1" (không biết nhóm nào là gì, nhưng không cần). `a == b` ⇔ số 0 bằng số 1 ⇒ `-1`. Ngược lại, nhóm lớn hơn là giá trị đa số; `3` thuộc nhóm A, `k` thuộc nhóm B ⇒ trả về đúng.

**Ngân sách.** `1 + (n − 4) + 3 = n` truy vấn (bản nộp) hoặc `n + 1` (doocs), đều `≤ 2n`. Mọi truy vấn có chỉ số tăng dần hợp lệ vì `n ≥ 5`.

## 11. Phân tích Time / Space

Gọi `n = reader.length()`.

| Cách | Số truy vấn | Time | Space |
|------|-------------|------|-------|
| 1. Mọi bộ 4 + vét cạn | `C(n,4)` | `O(2ⁿ · n⁴)` | `O(n⁴)` |
| 2. Doocs | `n + 1` | `O(n)` | `O(1)` |
| 3. Dùng lại truy vấn + nhãn | `n` | `O(n)` | `O(n)` |
| Bản nộp (mục 9) | `n` | `O(n)` | `O(1)` |
| Bài tập 5 (suy luận thêm) | `n − 1` | `O(n)` | `O(1)` |

Cận dưới thông tin: cần xác định quan hệ của `n − 1` phần tử với một phần tử tham chiếu ở mức đủ để đếm; mỗi truy vấn trả một trong 3 giá trị ⇒ số truy vấn tối thiểu cỡ `Θ(n)`.

## 12. Bộ test

Vì có **nhiều đáp án đúng**, mọi test dùng `valid_answer`. Gồm: ví dụ đề; edge case `n = 5` (mọi `2⁵` mảng), `n = 6, 7, 8` (mọi mảng — vét cạn toàn bộ), mảng hoà; mảng ngẫu nhiên seed cố định tới `n = 10⁵`; kiểm tra số truy vấn ≤ `2n`.

In [16]:
GOOD = [SolutionDoocs, SolutionCached, Solution]
for nums in examples:
    for S in GOOD:
        ans, calls, status = run(S, nums)
        assert status == "HỢP LỆ", (S.__name__, nums, ans, status)
print("ví dụ đề: OK")

total = 0
for n in range(5, 13):
    for bits in itertools.product([0, 1], repeat=n):
        nums = list(bits)
        for S in GOOD:
            ans, calls, status = run(S, nums)
            assert status == "HỢP LỆ" and calls <= 2 * n, (S.__name__, nums, ans, calls, status)
        total += 1
print(f"Mọi mảng nhị phân độ dài 5..12 ({total} mảng): mọi cách hợp lệ, ≤ 2n truy vấn")

ví dụ đề: OK


Mọi mảng nhị phân độ dài 5..12 (8160 mảng): mọi cách hợp lệ, ≤ 2n truy vấn


In [17]:
# Baseline vét cạn (không giới hạn truy vấn) cũng hợp lệ trên mọi mảng n = 5..7
for n in range(5, 8):
    for bits in itertools.product([0, 1], repeat=n):
        assert run(SolutionBrute, list(bits), limit=10**9)[2] == "HỢP LỆ"
print("Baseline vét cạn: hợp lệ trên mọi mảng n = 5..7")

Baseline vét cạn: hợp lệ trên mọi mảng n = 5..7


In [18]:
rng = random.Random(1538)
calls_seen = {S.__name__: set() for S in GOOD}
t0 = time.perf_counter()
for it in range(60):
    n = rng.choice([rng.randint(5, 30), rng.randint(5, 2000), rng.randint(90000, 100000)])
    p = rng.choice([0.5, 0.3, 0.7, 0.5])
    nums = [1 if rng.random() < p else 0 for _ in range(n)]
    if it % 10 == 0:                              # thỉnh thoảng ép hoà
        n -= n % 2
        nums = [0] * (n // 2) + [1] * (n // 2)
        rng.shuffle(nums)
    for S in GOOD:
        ans, calls, status = run(S, nums)
        assert status == "HỢP LỆ", (S.__name__, n, status)
        calls_seen[S.__name__].add(calls - n)
print("60 mảng ngẫu nhiên (seed 1538): OK | số truy vấn - n:", {k: sorted(v) for k, v in calls_seen.items()},
      f"| {time.perf_counter() - t0:.2f}s")

60 mảng ngẫu nhiên (seed 1538): OK | số truy vấn - n: {'SolutionDoocs': [1], 'SolutionCached': [0], 'Solution': [0]} | 2.16s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nghĩ `query = 4` nghĩa là "bốn số 1"

Kết quả 4 có thể là bốn số **0**. Suy ra giá trị tuyệt đối từ `query` là bất khả (mục 3.2).

In [19]:
class WrongAbsolute:
    def guessMajority(self, reader):
        n = reader.length()
        # SAI: coi query(0,1,2,3) == 4 nghĩa là các phần tử đầu là 1 và đa số là 1
        r = reader.query(0, 1, 2, 3)
        return 0 if r == 4 else -1

for nums in ([0, 0, 0, 0, 1], [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]):
    print(nums, "->", run(WrongAbsolute, nums))

[0, 0, 0, 0, 1] -> (0, 1, 'HỢP LỆ')
[0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1] -> (0, 1, 'SAI')


Sửa: chỉ so sánh **kết quả với kết quả** (cùng/khác), không suy ra giá trị bit.

### Lỗi 2 — Quên phân loại ba chỉ số `0, 1, 2`

Chỉ đếm các chỉ số `3..n−1` ⇒ thiếu 3 phần tử, kết luận sai khi chúng quyết định đa số.

In [20]:
class WrongSkipFirstThree:
    def guessMajority(self, reader):
        n = reader.length()
        x = reader.query(0, 1, 2, 3)
        a, b, k = 1, 0, -1
        for i in range(4, n):
            if reader.query(0, 1, 2, i) == x:
                a += 1
            else:
                b += 1; k = i
        if a == b:                               # SAI: chưa tính 0, 1, 2
            return -1
        return 3 if a > b else k

for nums in ([1, 1, 1, 0, 0], [1, 1, 0, 0, 1, 0, 1], [0, 0, 1, 1, 0]):
    print(nums, "->", run(WrongSkipFirstThree, nums))

[1, 1, 1, 0, 0] -> (3, 2, 'SAI')
[1, 1, 0, 0, 1, 0, 1] -> (-1, 4, 'SAI')
[0, 0, 1, 1, 0] -> (-1, 2, 'SAI')


Sửa: thêm ba truy vấn so `0, 1, 2` với 3 qua bộ cố định chứa chỉ số 4.

### Lỗi 3 — So `nums[0]` với `nums[3]` bằng `query(0, 1, 2, 0)`

Muốn "thay 3 bằng 0" trong bộ `{0,1,2,3}` nhưng 0 đã có trong bộ ⇒ chỉ số trùng, vi phạm `a < b < c < d`. Bộ giả lập báo "GỌI SAI".

In [21]:
class WrongDuplicateIndex:
    def guessMajority(self, reader):
        n = reader.length()
        x = reader.query(0, 1, 2, 3)
        a, b, k = 1, 0, -1
        for i in [0, 1, 2] + list(range(4, n)):
            if reader.query(0, 1, 2, i) == x:   # SAI khi i ∈ {0, 1, 2}
                a += 1
            else:
                b += 1; k = i
        return -1 if a == b else (3 if a > b else k)

print(run(WrongDuplicateIndex, examples[0]))

(None, 1, 'GỌI SAI: tham số sai: query(0, 1, 2, 0) với n=8')


Sửa: dùng bộ cố định **khác** (`{1,2,4}`, `{0,2,4}`, `{0,1,4}`) để so 0, 1, 2 với 3.

### Lỗi 4 — Quên trường hợp hoà

Trả về `3` hoặc `k` dù `a == b` ⇒ sai với ví dụ 3.

In [22]:
class WrongNoTie:
    def guessMajority(self, reader):
        ans = Solution().guessMajority(reader)
        return 3 if ans == -1 else ans           # SAI: bỏ mất -1 khi hoà (tương đương dùng a >= b)

for nums in examples:
    print(nums, "->", run(WrongNoTie, nums))

[0, 0, 1, 0, 1, 1, 1, 1] -> (2, 8, 'HỢP LỆ')
[0, 0, 1, 1, 0] -> (1, 5, 'HỢP LỆ')
[1, 0, 1, 0, 1, 0, 1, 0] -> (3, 8, 'SAI')


Sửa: kiểm tra `a == b` **trước** rồi mới so `a > b`.

### Lỗi 5 — Hỏi quá nhiều: so mỗi cặp phần tử bằng nhiều truy vấn

Ví dụ với mỗi `i` dùng **ba** truy vấn để "chắc ăn" ⇒ `3n` truy vấn, vượt `2n`.

In [23]:
class WrongTooMany:
    def guessMajority(self, reader):
        n = reader.length()
        a, b, k = 1, 0, -1
        x = reader.query(0, 1, 2, 3)
        for i in range(4, n):
            r1 = reader.query(0, 1, 2, i)
            r2 = reader.query(0, 1, 2, i)        # SAI: hỏi lặp vô ích
            r3 = reader.query(0, 1, 2, i)
            if r1 == x:
                a += 1
            else:
                b += 1; k = i
        return -1 if a == b else (3 if a > b else k)

print(run(WrongTooMany, [0, 1] * 10 + [1]))

(None, 43, 'VƯỢT GIỚI HẠN: vượt quá 42 lần gọi query')


Sửa: mỗi chỉ số chỉ cần **một** truy vấn.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Truy vấn trả về một **hàm đối xứng** của các phần tử ⇒ không biết giá trị tuyệt đối, chỉ biết **quan hệ** ⇒ đưa về bài **phân nhóm** so với một phần tử tham chiếu.
- Kỹ thuật **"thay một phần tử"**: hai truy vấn chỉ khác một chỉ số ⇒ so sánh trực tiếp hai phần tử đó (nếu hàm phân biệt được hai giá trị liền nhau).
- Bài hỏi "đa số" mà chỉ có hai giá trị ⇒ chỉ cần đếm kích thước hai nhóm và nhớ một đại diện mỗi nhóm.
- Bài nhiều đáp án ⇒ viết `valid_answer` trước khi test.
- Bài tương tự: *Find the Index of the Large Integer* (1533), *Majority Element* (169 — Boyer–Moore), *Guess Number Higher or Lower* (374), *Find the Celebrity* (277).

## 15. Biến thể của bài

1. **Trả về số lần xuất hiện của giá trị đa số**: `max(a, b)`.
2. **Trả về toàn bộ nhóm đa số**: lưu nhãn `same[i]` (cách 3) rồi lọc.
3. **Truy vấn 3 phần tử** `query3(a, b, c)` trả 3 nếu cùng giá trị, 1 nếu không: kỹ thuật thay phần tử vẫn dùng được nhưng `f(c) = 3 if c in (0,3) else 1` **không** phân biệt `1` với `2` ⇒ cần thiết kế lại.
4. **Tối thiểu số truy vấn** (follow-up): suy luận chỉ số thứ ba từ `x` ⇒ `n − 1` truy vấn (bài tập 5).
5. **Giá trị nhiều hơn 2 loại**: phân bố 4 phần tử không còn đủ thông tin để phân nhóm bằng một truy vấn mỗi phần tử.

In [24]:
def f3(c):
    return 3 if c in (0, 3) else 1

print("Biến thể 3 phần tử, bảng f3:", {c: f3(c) for c in range(4)},
      "| c=1 và c=2 cho cùng kết quả:", f3(1) == f3(2), "-> kỹ thuật thay phần tử thất bại khi s + bit chuyển 1↔2")

Biến thể 3 phần tử, bảng f3: {0: 3, 1: 1, 2: 1, 3: 3} | c=1 và c=2 cho cùng kết quả: True -> kỹ thuật thay phần tử thất bại khi s + bit chuyển 1↔2


## 16. So sánh các cách

| Cách | Ý tưởng | Truy vấn | Time | Space | Khi nào dùng |
|------|---------|----------|------|-------|--------------|
| 1. Mọi bộ 4 + vét cạn | hỏi hết rồi giải ngược | `C(n,4)` | `O(2ⁿ n⁴)` | `O(n⁴)` | kiểm chứng `n` rất nhỏ |
| 2. Doocs | thay phần tử, so với phần tử 3 | `n + 1` | `O(n)` | `O(1)` | lời giải chuẩn |
| 3. Dùng lại truy vấn + nhãn | như 2, bỏ truy vấn trùng | `n` | `O(n)` | `O(n)` | cần danh sách nhóm |
| 4. Khôi phục tới mức đảo bit | nhãn 0/1 so với phần tử 3 | `n` | `O(n)` | `O(n)` | hiểu bài / viết test |
| Bản nộp | 3 nhưng `O(1)` bộ nhớ | `n` | `O(n)` | `O(1)` | **nộp LeetCode** |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** Với `nums = [0,0,1,0,1,1,1,1]`, các truy vấn `query(0,1,2,3)`, `query(4,5,6,7)`, `query(0,2,4,5)`, `query(0,1,4,5)` trả về gì?

**Bài 2 (tính tay):** Chạy tay lời giải doocs trên `nums = [1,1,1,0,0,0,1]`: giá trị `x`, `y`, `a`, `b`, `k` cuối cùng và đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `ex3_same_as_3(reader, i, x)` trả `True` nếu `nums[i] == nums[3]` (với `i ≥ 4`, `x = query(0,1,2,3)`).

**Bài 4 (tự viết lại):** Viết `ex4_groups(reader)` trả về hai danh sách chỉ số `(nhóm cùng nums[3], nhóm khác)` với tối đa `n` truy vấn.

**Bài 5 (mini challenge — follow-up):** Viết lớp `Ex5MinQueries` với `guessMajority` dùng **đúng `n − 1` truy vấn**. Gợi ý: phân loại chỉ số `0` và `1` bằng truy vấn, còn chỉ số `2` thì **suy ra** từ `x = query(0,1,2,3)`.

In [25]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_same_as_3(reader, i: int, x: int) -> bool:
    # TODO: một truy vấn duy nhất, so với x
    raise NotImplementedError

In [26]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_groups(reader):
    # TODO
    raise NotImplementedError

class Ex5MinQueries:
    def guessMajority(self, reader: "ArrayReader") -> int:
        # TODO: đúng n - 1 truy vấn
        raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đếm số bit 1 trong 4 phần tử. (2) Tra bảng `f`: 0/4 → 4, 1/3 → 2, 2 → 0. (3) Lần lượt có 1, 4, 3, 2 bit 1.

**Bài 2.** (1) `s = nums[0]+nums[1]+nums[2] = 3`, `x = f(3 + 0) = 2`. (2) `i = 4, 5` khác `x`?: `f(3+0) = 2 = x` → cùng; `i = 6`: `f(4) = 4 ≠ x` → khác. (3) Ba truy vấn cuối: chỉ số 0, 1, 2 đều khác `nums[3]` ⇒ `a = 3`, `b = 4`, `k = 2` ⇒ đáp án 2.

**Bài 3.** (1) Giữ `{0,1,2}`, thay 3 bằng `i`. (2) `reader.query(0, 1, 2, i)`. (3) So bằng `== x`.

**Bài 4.** (1) Dùng lại `query(0,1,2,4)` làm `y`. (2) Ba truy vấn thay 0/1/2 bằng 3. (3) Gom chỉ số vào hai danh sách.

**Bài 5.** (1) Có `m0 = [nums[0] ≠ nums[3]]`, `m1 = [nums[1] ≠ nums[3]]` từ hai truy vấn. (2) Số phần tử trong `{0,1,2}` khác `nums[3]` là `m = m0 + m1 + m2`, và `x = f(...)` phụ thuộc `m`: `m = 0 → 4`, `1 → 2`, `2 → 0`, `3 → 2`. (3) Với `m0 + m1` đã biết, hai khả năng của `m2` luôn cho `x` khác nhau ⇒ suy ra `m2`.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
rd = ArrayReader([0, 0, 1, 0, 1, 1, 1, 1])
a1 = [rd.query(0, 1, 2, 3), rd.query(4, 5, 6, 7), rd.query(0, 2, 4, 5), rd.query(0, 1, 4, 5)]
print("Bài 1:", a1)
assert a1 == [2, 4, 2, 0]

# Bài 2
nums2 = [1, 1, 1, 0, 0, 0, 1]
ans2 = SolutionDoocs().guessMajority(ArrayReader(nums2))
print("Bài 2: đáp án", ans2, "| hợp lệ:", valid_answer(nums2, ans2))
assert ans2 == 2 and valid_answer(nums2, ans2)

Bài 1: [2, 4, 2, 0]
Bài 2: đáp án 2 | hợp lệ: True


In [28]:
# Bài 3
def ex3_same_as_3_sol(reader, i, x):
    return reader.query(0, 1, 2, i) == x

rng3 = random.Random(3)
for _ in range(200):
    nums = [rng3.randint(0, 1) for _ in range(rng3.randint(5, 40))]
    rd = ArrayReader(nums)
    x = rd.query(0, 1, 2, 3)
    for i in range(4, len(nums)):
        assert ex3_same_as_3_sol(rd, i, x) == (nums[i] == nums[3])
print("Bài 3: OK")

Bài 3: OK


In [29]:
# Bài 4
def ex4_groups_sol(reader):
    n = reader.length()
    x = reader.query(0, 1, 2, 3)
    same = {3: True}
    y = None
    for i in range(4, n):
        r = reader.query(0, 1, 2, i)
        if i == 4:
            y = r
        same[i] = r == x
    same[0] = reader.query(1, 2, 3, 4) == y
    same[1] = reader.query(0, 2, 3, 4) == y
    same[2] = reader.query(0, 1, 3, 4) == y
    A = sorted(i for i in same if same[i])
    B = sorted(i for i in same if not same[i])
    return A, B

for n in range(5, 11):
    for bits in itertools.product([0, 1], repeat=n):
        rd = ArrayReader(list(bits))
        A, B = ex4_groups_sol(rd)
        assert rd.calls <= n
        assert A == [i for i in range(n) if bits[i] == bits[3]] and B == [i for i in range(n) if bits[i] != bits[3]]
print("Bài 4: OK, ví dụ 1:", ex4_groups_sol(ArrayReader(examples[0])))

Bài 4: OK, ví dụ 1: ([0, 1, 3], [2, 4, 5, 6, 7])


In [30]:
# Bài 5
class Ex5MinQueriesSol:
    def guessMajority(self, reader):
        n = reader.length()
        x = reader.query(0, 1, 2, 3)
        a, b, k = 1, 0, -1
        y = None
        for i in range(4, n):
            r = reader.query(0, 1, 2, i)
            if i == 4:
                y = r
            if r == x:
                a += 1
            else:
                b += 1; k = i
        m0 = reader.query(1, 2, 3, 4) != y        # nums[0] khác nums[3]?
        m1 = reader.query(0, 2, 3, 4) != y        # nums[1] khác nums[3]?
        fm = {0: 4, 1: 2, 2: 0, 3: 2}             # x theo số phần tử của {0,1,2} khác nums[3]
        m2 = fm[m0 + m1 + 1] == x and fm[m0 + m1] != x
        for idx, m in ((0, m0), (1, m1), (2, m2)):
            if m:
                b += 1; k = idx
            else:
                a += 1
        if a == b:
            return -1
        return 3 if a > b else k

for n in range(5, 13):
    for bits in itertools.product([0, 1], repeat=n):
        ans, calls, status = run(Ex5MinQueriesSol, list(bits))
        assert status == "HỢP LỆ" and calls == n - 1, (bits, ans, calls)
print("Bài 5: OK — luôn đúng n - 1 truy vấn trên mọi mảng n = 5..12")

Bài 5: OK — luôn đúng n - 1 truy vấn trên mọi mảng n = 5..12


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def guessMajority(self, reader):
        n = reader.length()
        x = reader.query(0, 1, 2, 3)              # tham chiếu: phần tử 3
        a, b, k, y = 1, 0, -1, None
        for i in range(4, n):                     # thay 3 bằng i trong bộ {0,1,2,·}
            r = reader.query(0, 1, 2, i)
            if i == 4: y = r
            if r == x: a += 1
            else: b += 1; k = i
        for idx, q in ((0, (1,2,3,4)), (1, (0,2,3,4)), (2, (0,1,3,4))):
            if reader.query(*q) == y: a += 1      # thay idx bằng 3 trong bộ {0,1,2,4}
            else: b += 1; k = idx
        return -1 if a == b else (3 if a > b else k)
```

**Template "truy vấn đối xứng → phân nhóm":** chọn phần tử tham chiếu; so từng phần tử với nó bằng hai truy vấn chỉ khác một chỉ số; đếm hai nhóm; trả đại diện.

**Checklist trước khi submit:**
- [ ] Mọi truy vấn có chỉ số **tăng dần, phân biệt** (`a < b < c < d`).
- [ ] Đã phân loại cả `0, 1, 2` (không chỉ `4..n−1`).
- [ ] Hoà ⇒ `-1` (kiểm tra `a == b` trước).
- [ ] `k` luôn là chỉ số thuộc nhóm B khi `b > a`.
- [ ] Số truy vấn ≤ `2n` (ở đây `n` hoặc `n + 1`).